In [4]:
import pyterrier as pt
import pyt_splade
import torch
import os
import warnings

torch.manual_seed(26)  # for reproducibility


def detect_device():
    if torch.cuda.is_available():
        return "cuda"
    elif torch.backends.mps.is_available():
        return "mps"
    else:
        return "cpu"


device = detect_device()


In [2]:
splade = pyt_splade.Splade(model="C:/splade_model", device=device)

HFValidationError: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: 'C:/splade_model'.

In [ ]:
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.autocast.*")

splade = pyt_splade.Splade(model="C:/splade_model", device="cuda", max_length=256)
index_dir = os.path.abspath("./msmarco_splade_index")
dataset = pt.get_dataset("irds:msmarco-passage")
indexer = pt.IterDictIndexer(index_dir, pretokenised=True, overwrite=True)
indexing_pipeline = splade.doc_encoder(batch_size=64) >> indexer

index_ref = indexing_pipeline.index(dataset.get_corpus_iter(), batch_size=64)

msmarco-passage documents:   0%|          | 0/8841823 [00:00<?, ?it/s]

In [ ]:
dataset = pt.get_dataset("msmarco_passage")

index_ref = dataset.get_index("terrier_stemmed")
bm25 = pt.terrier.Retriever(index_ref, wmodel="BM25", verbose=True)

topics = dataset.get_topics('dev.small')
topics['query'] = topics['query'].str.replace(r'[\?\:\!]', '', regex=True)
qrels = dataset.get_qrels('dev.small')

experiment = pt.Experiment(
    [bm25],
    topics,
    qrels,
    [nDCG @ 10, MAP, Recall @ 100]
)

print(experiment)

data.direct.bf: 100%|██████████| 486M/486M [01:00<00:00, 8.49MiB/s] 
data.document.fsarrayfile: 100%|██████████| 177M/177M [00:26<00:00, 6.93MiB/s] 
data.inverted.bf: 100%|██████████| 377M/377M [00:48<00:00, 8.10MiB/s] 
data.lexicon.fsomapfile: 100%|██████████| 100M/100M [00:09<00:00, 11.5MiB/s] 
data.lexicon.fsomaphash: 100%|██████████| 0.99k/0.99k [00:00<00:00, 63.6kiB/s]
data.lexicon.fsomapid: 100%|██████████| 4.47M/4.47M [00:00<00:00, 5.11MiB/s]
data.meta.idx: 100%|██████████| 67.5M/67.5M [00:08<00:00, 8.31MiB/s]
data.meta.zdata: 100%|██████████| 193M/193M [00:28<00:00, 7.05MiB/s] 
data.properties: 100%|██████████| 4.29k/4.29k [00:00<00:00, 616kiB/s]
md5sums: 100%|██████████| 480/480 [00:00<00:00, 5.50MiB/s]
Java started (triggered by Retriever.__init__) and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


collectionandqueries.tar.gz:   6%|▌         | 55.6M/0.99G [02:50<2:11:12, 127kiB/s]

In [ ]:
import sys

sys.executable

In [ ]:
!python -c "from transformers import pipeline; print(pipeline('sentiment-analysis')('hugging face is the best'))"